In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Display settings
pd.set_option('display.max_columns', None)
plt.style.use('ggplot')

print("Environment ready ✅")

Matplotlib is building the font cache; this may take a moment.


Environment ready ✅


In [5]:
df = pd.read_csv("data/results.csv")

In [6]:
print(df.shape)
df.head(10)

(49498, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False
5,1876-03-25,Scotland,Wales,4.0,0.0,Friendly,Glasgow,Scotland,False
6,1877-03-03,England,Scotland,1.0,3.0,Friendly,London,England,False
7,1877-03-05,Wales,Scotland,0.0,2.0,Friendly,Wrexham,Wales,False
8,1878-03-02,Scotland,England,7.0,2.0,Friendly,Glasgow,Scotland,False
9,1878-03-23,Scotland,Wales,9.0,0.0,Friendly,Glasgow,Scotland,False


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49498 entries, 0 to 49497
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49498 non-null  str    
 1   home_team   49498 non-null  str    
 2   away_team   49498 non-null  str    
 3   home_score  49487 non-null  float64
 4   away_score  49487 non-null  float64
 5   tournament  49498 non-null  str    
 6   city        49498 non-null  str    
 7   country     49498 non-null  str    
 8   neutral     49498 non-null  bool   
dtypes: bool(1), float64(2), str(6)
memory usage: 3.1 MB


In [8]:
df['date'] = pd.to_datetime(df['date'])
print(df['date'].min(), "to", df['date'].max())

1872-11-30 00:00:00 to 2026-07-06 00:00:00


In [9]:
df['tournament'].value_counts().head(20)

tournament
Friendly                                18388
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1057
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
CFU Caribbean Cup qualification           606
Merdeka Tournament                        599
British Home Championship                 523
CONCACAF Nations League                   422
AFC Asian Cup                             421
Gold Cup                                  420
Gulf Cup                                  410
Island Games                              394
UEFA Euro                                 388
Asian Games                               368
Name: count, dtype: int64

In [10]:
df.isnull().sum()

date           0
home_team      0
away_team      0
home_score    11
away_score    11
tournament     0
city           0
country        0
neutral        0
dtype: int64

In [12]:
df = df.dropna(subset=['home_score', 'away_score'])


In [13]:
df.isnull().sum()

date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64

In [14]:
# Just the actual World Cup tournament matches (not qualifiers)
wc_df = df[df['tournament'] == 'FIFA World Cup']
print(wc_df.shape)
wc_df.head()

(1046, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
1490,1930-07-13,Belgium,United States,0.0,3.0,FIFA World Cup,Montevideo,Uruguay,True
1491,1930-07-13,France,Mexico,4.0,1.0,FIFA World Cup,Montevideo,Uruguay,True
1492,1930-07-14,Brazil,Yugoslavia,1.0,2.0,FIFA World Cup,Montevideo,Uruguay,True
1493,1930-07-14,Peru,Romania,1.0,3.0,FIFA World Cup,Montevideo,Uruguay,True
1494,1930-07-15,Argentina,France,1.0,0.0,FIFA World Cup,Montevideo,Uruguay,True


In [15]:
wc_df = df[df['tournament'] == 'FIFA World Cup']
wc_df['neutral'].value_counts()

neutral
True     914
False    132
Name: count, dtype: int64

In [16]:
# Create two rows per match — one from each team's perspective
home_df = wc_df.copy()
home_df['team'] = home_df['home_team']
home_df['opponent'] = home_df['away_team']
home_df['goals_for'] = home_df['home_score']
home_df['goals_against'] = home_df['away_score']
home_df['is_home_label'] = ~home_df['neutral']  # only True for actual host matches

away_df = wc_df.copy()
away_df['team'] = away_df['away_team']
away_df['opponent'] = away_df['home_team']
away_df['goals_for'] = away_df['away_score']
away_df['goals_against'] = away_df['home_score']
away_df['is_home_label'] = False  # away side is never the host

team_matches = pd.concat([home_df, away_df], ignore_index=True)
team_matches['win'] = (team_matches['goals_for'] > team_matches['goals_against']).astype(int)

print(team_matches.shape)
team_matches.head()

(2092, 15)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,team,opponent,goals_for,goals_against,is_home_label,win
0,1930-07-13,Belgium,United States,0.0,3.0,FIFA World Cup,Montevideo,Uruguay,True,Belgium,United States,0.0,3.0,False,0
1,1930-07-13,France,Mexico,4.0,1.0,FIFA World Cup,Montevideo,Uruguay,True,France,Mexico,4.0,1.0,False,1
2,1930-07-14,Brazil,Yugoslavia,1.0,2.0,FIFA World Cup,Montevideo,Uruguay,True,Brazil,Yugoslavia,1.0,2.0,False,0
3,1930-07-14,Peru,Romania,1.0,3.0,FIFA World Cup,Montevideo,Uruguay,True,Peru,Romania,1.0,3.0,False,0
4,1930-07-15,Argentina,France,1.0,0.0,FIFA World Cup,Montevideo,Uruguay,True,Argentina,France,1.0,0.0,False,1


In [17]:
team_matches[['team', 'goals_for', 'goals_against', 'win']].head(10)

,team,goals_for,goals_against,win
0,Belgium,0.0,3.0,0
1,France,4.0,1.0,1
2,Brazil,1.0,2.0,0
3,Peru,1.0,3.0,0
4,Argentina,1.0,0.0,1
5,Chile,3.0,0.0,1
6,Bolivia,0.0,4.0,0
7,Paraguay,0.0,3.0,0
8,Uruguay,1.0,0.0,1
9,Argentina,6.0,3.0,1


In [18]:
team_matches = team_matches.sort_values(['team', 'date']).reset_index(drop=True)
team_matches.head(10)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,team,opponent,goals_for,goals_against,is_home_label,win
0,1982-06-16,Germany,Algeria,1.0,2.0,FIFA World Cup,Gijón,Spain,True,Algeria,Germany,2.0,1.0,False,1
1,1982-06-21,Algeria,Austria,0.0,2.0,FIFA World Cup,Oviedo,Spain,True,Algeria,Austria,0.0,2.0,False,0
2,1982-06-24,Algeria,Chile,3.0,2.0,FIFA World Cup,Oviedo,Spain,True,Algeria,Chile,3.0,2.0,False,1
3,1986-06-03,Algeria,Northern Ireland,1.0,1.0,FIFA World Cup,Guadalajara,Mexico,True,Algeria,Northern Ireland,1.0,1.0,False,0
4,1986-06-06,Brazil,Algeria,1.0,0.0,FIFA World Cup,Guadalajara,Mexico,True,Algeria,Brazil,0.0,1.0,False,0
5,1986-06-12,Algeria,Spain,0.0,3.0,FIFA World Cup,Monterrey,Mexico,True,Algeria,Spain,0.0,3.0,False,0
6,2010-06-13,Algeria,Slovenia,0.0,1.0,FIFA World Cup,Polokwane,South Africa,True,Algeria,Slovenia,0.0,1.0,False,0
7,2010-06-18,England,Algeria,0.0,0.0,FIFA World Cup,Cape Town,South Africa,True,Algeria,England,0.0,0.0,False,0
8,2010-06-23,United States,Algeria,1.0,0.0,FIFA World Cup,Pretoria,South Africa,True,Algeria,United States,0.0,1.0,False,0
9,2014-06-17,Belgium,Algeria,2.0,1.0,FIFA World Cup,Belo Horizonte,Brazil,True,Algeria,Belgium,1.0,2.0,False,0


In [19]:
# Rolling form over last 5 matches, shifted by 1 to avoid leakage
# (shift ensures we only use matches BEFORE the current one)
team_matches['form_5'] = (
    team_matches.groupby('team')['win']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

team_matches['goals_avg_5'] = (
    team_matches.groupby('team')['goals_for']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

team_matches['conceded_avg_5'] = (
    team_matches.groupby('team')['goals_against']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

team_matches[['team', 'date', 'win', 'form_5', 'goals_avg_5', 'conceded_avg_5']].head(15)

,team,date,win,form_5,goals_avg_5,conceded_avg_5
0,Algeria,1982-06-16,1,NaN,NaN,NaN
1,Algeria,1982-06-21,0,1.000000,2.000000,1.000000
2,Algeria,1982-06-24,1,0.500000,1.000000,1.500000
3,Algeria,1986-06-03,0,0.666667,1.666667,1.666667
4,Algeria,1986-06-06,0,0.500000,1.500000,1.500000
5,Algeria,1986-06-12,0,0.400000,1.200000,1.400000
6,Algeria,2010-06-13,0,0.200000,0.800000,1.800000
7,Algeria,2010-06-18,0,0.200000,0.800000,1.600000
8,Algeria,2010-06-23,0,0.000000,0.200000,1.200000
9,Algeria,2014-06-17,0,0.000000,0.000000,1.200000


In [20]:
team_matches_clean = team_matches.dropna(subset=['form_5', 'goals_avg_5', 'conceded_avg_5'])
print(team_matches_clean.shape)

(2006, 18)


In [21]:
team_matches['is_host'] = team_matches['is_home_label'].astype(int)
team_matches[['team', 'date', 'is_host']].value_counts('is_host')

is_host
0    1960
1     132
Name: count, dtype: int64

In [22]:
def h2h_winrate(row, history):
    past = history[
        (history['team'] == row['team']) &
        (history['opponent'] == row['opponent']) &
        (history['date'] < row['date'])
    ]
    if len(past) == 0:
        return np.nan
    return past['win'].mean()

team_matches['h2h_winrate'] = team_matches.apply(
    lambda row: h2h_winrate(row, team_matches), axis=1
)

In [23]:
team_matches[['team', 'opponent', 'date', 'h2h_winrate']].head(10)

,team,opponent,date,h2h_winrate
0,Algeria,Germany,1982-06-16,NaN
1,Algeria,Austria,1982-06-21,NaN
2,Algeria,Chile,1982-06-24,NaN
3,Algeria,Northern Ireland,1986-06-03,NaN
4,Algeria,Brazil,1986-06-06,NaN
5,Algeria,Spain,1986-06-12,NaN
6,Algeria,Slovenia,2010-06-13,NaN
7,Algeria,England,2010-06-18,NaN
8,Algeria,United States,2010-06-23,NaN
9,Algeria,Belgium,2014-06-17,NaN


In [24]:
print(team_matches['h2h_winrate'].isnull().mean())


0.6577437858508605


In [25]:
feature_cols = ['form_5', 'goals_avg_5', 'conceded_avg_5', 'is_host']

model_df = team_matches.dropna(subset=feature_cols + ['win']).copy()
print(model_df.shape)

(2006, 20)


In [26]:
X = model_df[feature_cols].to_numpy(dtype=float)
y = model_df['win'].to_numpy(dtype=float)

print(X.shape, y.shape)
print("Class balance:", y.mean())

(2006, 4) (2006,)
Class balance: 0.3898305084745763


In [29]:
def train_test_split_stratified(X, y, test_size=0.2, seed=42):
    rng = np.random.default_rng(seed)
    idx_pos = np.where(y == 1)[0]
    idx_neg = np.where(y == 0)[0]
    rng.shuffle(idx_pos)
    rng.shuffle(idx_neg)
    n_pos_test = int(len(idx_pos) * test_size)
    n_neg_test = int(len(idx_neg) * test_size)
    test_idx = np.concatenate([idx_pos[:n_pos_test], idx_neg[:n_neg_test]])
    train_idx = np.setdiff1d(np.arange(len(y)), test_idx)
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

In [30]:
X_train, X_test, y_train, y_test = train_test_split_stratified(X, y, test_size=0.2, seed=42)
print(X_train.shape, X_test.shape)

(1606, 4) (400, 4)


In [31]:
def standard_scale(X_train, X_test):
    mu = X_train.mean(axis=0)
    sigma = X_train.std(axis=0)
    sigma[sigma == 0] = 1.0
    return (X_train - mu) / sigma, (X_test - mu) / sigma, mu, sigma

X_train_s, X_test_s, mu, sigma = standard_scale(X_train, X_test)
print("Train mean (should be ~0):", X_train_s.mean(axis=0))
print("Train std (should be ~1):", X_train_s.std(axis=0))

Train mean (should be ~0): [-2.03642278e-15  2.99677261e-17 -1.82647587e-15 -1.32936418e-16]
Train std (should be ~1): [1. 1. 1. 1.]


In [32]:
class LogisticRegressionScratch:
    def __init__(self, learning_rate=0.05, n_iterations=10_000,
                 tol=1e-6, l2_lambda=0.0, verbose=False):
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.tol = tol
        self.l2_lambda = l2_lambda
        self.verbose = verbose
        self.w = None
        self.b = 0.0
        self.loss_history = []

    @staticmethod
    def _sigmoid(z):
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def _loss(self, y, p):
        eps = 1e-12
        p = np.clip(p, eps, 1 - eps)
        ce = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
        reg = (self.l2_lambda / (2 * len(y))) * np.sum(self.w ** 2)
        return ce + reg

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0.0

        prev_loss = np.inf
        for i in range(self.n_iterations):
            z = X @ self.w + self.b
            p = self._sigmoid(z)

            error = p - y
            grad_w = (X.T @ error) / n_samples + (self.l2_lambda / n_samples) * self.w
            grad_b = np.mean(error)

            self.w -= self.lr * grad_w
            self.b -= self.lr * grad_b

            loss = self._loss(y, p)
            self.loss_history.append(loss)
            if abs(prev_loss - loss) < self.tol:
                if self.verbose:
                    print(f"Converged at iteration {i}, loss={loss:.6f}")
                break
            prev_loss = loss
        return self

    def predict_proba(self, X):
        return self._sigmoid(X @ self.w + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

In [33]:
model = LogisticRegressionScratch(learning_rate=0.1, n_iterations=10000, verbose=True)
model.fit(X_train_s, y_train)

print("Final weights:", model.w)
print("Final bias:", model.b)
print("Number of iterations run:", len(model.loss_history))

Converged at iteration 237, loss=0.643668
Final weights: [ 0.1128769   0.20606532 -0.20415615  0.21736331]
Final bias: -0.46647881267320485
Number of iterations run: 238


In [34]:
# From-scratch metrics
def confusion_counts(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp, tn, fp, fn

def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def precision(y_true, y_pred):
    tp, _, fp, _ = confusion_counts(y_true, y_pred)
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

def recall(y_true, y_pred):
    tp, _, _, fn = confusion_counts(y_true, y_pred)
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def f1_score(y_true, y_pred):
    p, r = precision(y_true, y_pred), recall(y_true, y_pred)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

y_pred = model.predict(X_test_s)
y_proba = model.predict_proba(X_test_s)

print("Accuracy:", accuracy(y_test, y_pred))
print("Precision:", precision(y_test, y_pred))
print("Recall:", recall(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("\nBaseline (majority class) accuracy:", 1 - y_test.mean())

Accuracy: 0.6325
Precision: 0.6153846153846154
Recall: 0.15384615384615385
F1: 0.24615384615384617

Baseline (majority class) accuracy: 0.61


In [35]:
def roc_auc(y_true, y_scores):
    order = np.argsort(y_scores)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(y_scores) + 1)
    n_pos = np.sum(y_true == 1)
    n_neg = np.sum(y_true == 0)
    if n_pos == 0 or n_neg == 0:
        return np.nan
    sum_pos_ranks = np.sum(ranks[y_true == 1])
    return (sum_pos_ranks - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

print("ROC-AUC:", roc_auc(y_test, y_proba))

ROC-AUC: 0.6221101303068516


In [36]:
for t in [0.3, 0.35, 0.4, 0.45, 0.5]:
    pred_t = model.predict(X_test_s, threshold=t)
    print(f"Threshold {t}: Accuracy={accuracy(y_test, pred_t):.3f}, "
          f"Precision={precision(y_test, pred_t):.3f}, "
          f"Recall={recall(y_test, pred_t):.3f}, "
          f"F1={f1_score(y_test, pred_t):.3f}")

Threshold 0.3: Accuracy=0.472, Precision=0.414, Recall=0.853, F1=0.558
Threshold 0.35: Accuracy=0.552, Precision=0.452, Recall=0.699, F1=0.549
Threshold 0.4: Accuracy=0.623, Precision=0.516, Recall=0.519, F1=0.518
Threshold 0.45: Accuracy=0.618, Precision=0.518, Recall=0.276, F1=0.360
Threshold 0.5: Accuracy=0.632, Precision=0.615, Recall=0.154, F1=0.246


In [37]:
rankings = pd.read_csv("data/fifa_ranking-2024-06-20.csv")
rankings['rank_date'] = pd.to_datetime(rankings['rank_date'])
rankings = rankings.sort_values(['country_full', 'rank_date']).reset_index(drop=True)
print(rankings.shape)
rankings.head()

(67472, 8)


,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,204.0,Afghanistan,AFG,7.0,0.0,204,AFC,2003-01-15
1,203.0,Afghanistan,AFG,9.0,7.0,-1,AFC,2003-02-19
2,198.0,Afghanistan,AFG,48.0,9.0,-5,AFC,2003-03-26
3,198.0,Afghanistan,AFG,48.0,48.0,0,AFC,2003-04-23
4,199.0,Afghanistan,AFG,48.0,48.0,1,AFC,2003-05-21


In [38]:
# merge_asof requires both frames sorted by the date column
team_matches_sorted = team_matches.sort_values('date').reset_index(drop=True)
rankings_sorted = rankings.sort_values('rank_date').reset_index(drop=True)

team_matches_ranked = pd.merge_asof(
    team_matches_sorted,
    rankings_sorted[['rank_date', 'country_full', 'rank', 'total_points']],
    left_on='date',
    right_on='rank_date',
    left_by='team',
    right_by='country_full',
    direction='backward'  # only use rankings published ON or BEFORE the match date — no leakage
)

print(team_matches_ranked.shape)
team_matches_ranked[['team', 'date', 'rank', 'total_points']].head(10)


(2092, 24)


,team,date,rank,total_points
0,Mexico,1930-07-13,NaN,NaN
1,Belgium,1930-07-13,NaN,NaN
2,United States,1930-07-13,NaN,NaN
3,France,1930-07-13,NaN,NaN
4,Romania,1930-07-14,NaN,NaN
5,Peru,1930-07-14,NaN,NaN
6,Brazil,1930-07-14,NaN,NaN
7,Yugoslavia,1930-07-14,NaN,NaN
8,France,1930-07-15,NaN,NaN
9,Argentina,1930-07-15,NaN,NaN


In [39]:
print(team_matches_ranked['rank'].isnull().sum())

1049


In [40]:
# How many nulls are simply because the match happened before rankings existed?
pre_ranking_era = team_matches_ranked[team_matches_ranked['date'] < '1992-12-31']
print("Matches before rankings existed:", pre_ranking_era.shape[0])

# Of the nulls, how many are from AFTER rankings started (true mismatches)?
post_ranking_era_nulls = team_matches_ranked[
    (team_matches_ranked['date'] >= '1992-12-31') &
    (team_matches_ranked['rank'].isnull())
]
print("Nulls that ARE mismatches (post-1992):", post_ranking_era_nulls.shape[0])
post_ranking_era_nulls[['team', 'date']].drop_duplicates('team')











































Matches before rankings existed: 928
Nulls that ARE mismatches (post-1992): 121


,team,date
931,South Korea,1994-06-17
936,United States,1994-06-18
1052,Iran,1998-06-14
1054,Serbia,1998-06-14
1182,China,2002-06-04
1293,Ivory Coast,2006-06-10
1306,Czech Republic,2006-06-12
1442,North Korea,2010-06-15
1946,Curaçao,2026-06-14
1953,Cape Verde,2026-06-15


In [41]:
# Search the rankings dataset for close matches to each mismatched name
mismatched_teams = post_ranking_era_nulls['team'].unique()

for team in mismatched_teams:
    matches = rankings[rankings['country_full'].str.contains(team.split()[-1], case=False, na=False)]
    print(f"'{team}' -> possible matches: {matches['country_full'].unique()}")

'South Korea' -> possible matches: <StringArray>
['Korea DPR', 'Korea Republic']
Length: 2, dtype: str
'United States' -> possible matches: <StringArray>
[]
Length: 0, dtype: str
'Iran' -> possible matches: <StringArray>
['IR Iran']
Length: 1, dtype: str
'Serbia' -> possible matches: <StringArray>
['Serbia', 'Serbia and Montenegro']
Length: 2, dtype: str
'China' -> possible matches: <StringArray>
['China PR']
Length: 1, dtype: str
'Ivory Coast' -> possible matches: <StringArray>
[]
Length: 0, dtype: str
'Czech Republic' -> possible matches: <StringArray>
['Central African Republic',       'Dominican Republic',
           'Korea Republic',          'Kyrgyz Republic',
      'Republic of Ireland']
Length: 5, dtype: str
'North Korea' -> possible matches: <StringArray>
['Korea DPR', 'Korea Republic']
Length: 2, dtype: str
'Curaçao' -> possible matches: <StringArray>
[]
Length: 0, dtype: str
'Cape Verde' -> possible matches: <StringArray>
['Cabo Verde']
Length: 1, dtype: str
'DR Congo' -> po

In [42]:
for name in ['USA', "Côte d'Ivoire", 'Czech Republic', 'Curacao']:
    print(name, '->', rankings['country_full'].str.contains(name, case=False, na=False).sum())

USA -> 333
Côte d'Ivoire -> 333
Czech Republic -> 0
Curacao -> 136


In [43]:
print(rankings[rankings['country_full'].str.contains('Czech', case=False, na=False)]['country_full'].unique())

<StringArray>
['Czechia', 'Czechoslovakia']
Length: 2, dtype: str


In [44]:
name_mapping = {
    'South Korea': 'Korea Republic',
    'North Korea': 'Korea DPR',
    'United States': 'USA',
    'Iran': 'IR Iran',
    'China': 'China PR',
    'Ivory Coast': "Côte d'Ivoire",
    'Czech Republic': 'Czechia',
    'Curaçao': 'Curacao',
    'Cape Verde': 'Cabo Verde',
    'DR Congo': 'Congo DR',
}

team_matches_sorted['team_mapped'] = team_matches_sorted['team'].replace(name_mapping)

team_matches_ranked = pd.merge_asof(
    team_matches_sorted,
    rankings_sorted[['rank_date', 'country_full', 'rank', 'total_points']],
    left_on='date',
    right_on='rank_date',
    left_by='team_mapped',
    right_by='country_full',
    direction='backward'
)

print("Total nulls after mapping fix:", team_matches_ranked['rank'].isnull().sum())

# Double-check: should now be close to the 928 pre-1992 matches, nothing more
remaining_nulls = team_matches_ranked

Total nulls after mapping fix: 932


In [45]:
remaining_nulls = team_matches_ranked[team_matches_ranked['rank'].isnull()]
still_broken = remaining_nulls[remaining_nulls['date'] >= '1992-12-31']
print(still_broken.shape[0])
still_broken[['team', 'date']]

4


,team,date
1054,Serbia,1998-06-14
1087,Serbia,1998-06-21
1112,Serbia,1998-06-25
1138,Serbia,1998-06-29


In [46]:
team_matches_final = team_matches_ranked.dropna(subset=['rank', 'total_points']).copy()
print(team_matches_final.shape)

(1160, 25)


In [47]:
team_matches_final['opponent_mapped'] = team_matches_final['opponent'].replace(name_mapping)

team_matches_final = pd.merge_asof(
    team_matches_final.sort_values('date'),
    rankings_sorted[['rank_date', 'country_full', 'rank']].rename(
        columns={'rank': 'opponent_rank', 'country_full': 'country_full_opp'}
    ),
    left_on='date',
    right_on='rank_date',
    left_by='opponent_mapped',
    right_by='country_full_opp',
    direction='backward'
)

print("Opponent rank nulls:", team_matches_final['opponent_rank'].isnull().sum())

Opponent rank nulls: 4


In [48]:
team_matches_final = team_matches_final.dropna(subset=['rank', 'opponent_rank']).copy()

# Lower FIFA rank number = better team, so flip the sign:
# positive rank_diff = opponent is worse ranked = should favor this team winning
team_matches_final['rank_diff'] = team_matches_final['opponent_rank'] - team_matches_final['rank']

print(team_matches_final.shape)
team_matches_final[['team', 'opponent', 'rank', 'opponent_rank', 'rank_diff']].head(10)

(1156, 30)


,team,opponent,rank,opponent_rank,rank_diff
0,Spain,South Korea,5.0,37.0,32.0
1,Bolivia,Germany,43.0,1.0,-42.0
2,Germany,Bolivia,1.0,43.0,42.0
3,South Korea,Spain,37.0,5.0,-32.0
4,Italy,Republic of Ireland,4.0,14.0,10.0
5,Republic of Ireland,Italy,14.0,4.0,-10.0
6,United States,Switzerland,23.0,12.0,-11.0
7,Romania,Colombia,7.0,17.0,10.0
8,Colombia,Romania,17.0,7.0,-10.0
9,Switzerland,United States,12.0,23.0,11.0


In [49]:
feature_cols_v2 = ['form_5', 'goals_avg_5', 'conceded_avg_5', 'is_host', 'rank_diff']

model_df_v2 = team_matches_final.dropna(subset=feature_cols_v2 + ['win']).copy()
print(model_df_v2.shape)

X2 = model_df_v2[feature_cols_v2].to_numpy(dtype=float)
y2 = model_df_v2['win'].to_numpy(dtype=float)

print(X2.shape, y2.shape)
print("Class balance:", y2.mean())

(1129, 30)
(1129, 5) (1129,)
Class balance: 0.3852967227635075


In [50]:
X2_train, X2_test, y2_train, y2_test = train_test_split_stratified(X2, y2, test_size=0.2, seed=42)
X2_train_s, X2_test_s, mu2, sigma2 = standard_scale(X2_train, X2_test)
print(X2_train_s.shape, X2_test_s.shape)

(904, 5) (225, 5)


In [51]:
model_v2 = LogisticRegressionScratch(learning_rate=0.1, n_iterations=10000, verbose=True)
model_v2.fit(X2_train_s, y2_train)

print("Feature order:", feature_cols_v2)
print("Final weights:", model_v2.w)
print("Final bias:", model_v2.b)

Converged at iteration 265, loss=0.595772
Feature order: ['form_5', 'goals_avg_5', 'conceded_avg_5', 'is_host', 'rank_diff']
Final weights: [0.08553233 0.15529436 0.07453158 0.17986969 0.77350062]
Final bias: -0.5361789697119193


In [52]:
np.corrcoef(model_df_v2['conceded_avg_5'], model_df_v2['rank_diff'])

array([[ 1.        , -0.29309716],
       [-0.29309716,  1.        ]])

In [53]:
y2_pred = model_v2.predict(X2_test_s)
y2_proba = model_v2.predict_proba(X2_test_s)

print("Accuracy:", accuracy(y2_test, y2_pred))
print("Precision:", precision(y2_test, y2_pred))
print("Recall:", recall(y2_test, y2_pred))
print("F1:", f1_score(y2_test, y2_pred))
print("ROC-AUC:", roc_auc(y2_test, y2_proba))
print("\nBaseline (majority class) accuracy:", 1 - y2_test.mean())

Accuracy: 0.7066666666666667
Precision: 0.6842105263157895
Recall: 0.4482758620689655
F1: 0.5416666666666666
ROC-AUC: 0.7559553556555055

Baseline (majority class) accuracy: 0.6133333333333333


In [54]:
def k_fold_cv(X, y, k=5, seed=42, **model_kwargs):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(y))
    folds = np.array_split(idx, k)
    results = []

    for i in range(k):
        val_idx = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])

        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        # Scale INSIDE the fold — fit only on this fold's training data
        X_tr_s, X_val_s, _, _ = standard_scale(X_tr, X_val)

        model = LogisticRegressionScratch(**model_kwargs).fit(X_tr_s, y_tr)
        proba = model.predict_proba(X_val_s)
        pred = (proba >= 0.5).astype(int)

        results.append({
            "fold": i + 1,
            "accuracy": accuracy(y_val, pred),
            "precision": precision(y_val, pred),
            "recall": recall(y_val, pred),
            "f1": f1_score(y_val, pred),
            "auc": roc_auc(y_val, proba),
        })
    return results

cv_results = k_fold_cv(X2, y2, k=5, learning_rate=0.1, n_iterations=10000)

import pandas as pd
cv_df = pd.DataFrame(cv_results)
print(cv_df)
print("\nMean ± Std across folds:")
print(cv_df[['accuracy', 'precision', 'recall', 'f1', 'auc']].agg(['mean', 'std']))

   fold  accuracy  precision    recall        f1       auc
0     1  0.725664   0.661017  0.481481  0.557143  0.729502
1     2  0.685841   0.639344  0.443182  0.523490  0.738966
2     3  0.690265   0.616667  0.440476  0.513889  0.727364
3     4  0.712389   0.776119  0.509804  0.615385  0.731736
4     5  0.684444   0.578947  0.412500  0.481752  0.705776

Mean ± Std across folds:
      accuracy  precision    recall        f1       auc
mean  0.699721   0.654419  0.457489  0.538332  0.726669
std   0.018364   0.074503  0.038179  0.050769  0.012469


In [55]:
lambda_values = [0, 0.01, 0.1, 1, 5, 10]
l2_results = []

for lam in lambda_values:
    cv_res = k_fold_cv(X2, y2, k=5, learning_rate=0.1, n_iterations=10000, l2_lambda=lam)
    cv_res_df = pd.DataFrame(cv_res)
    l2_results.append({
        "lambda": lam,
        "auc_mean": cv_res_df['auc'].mean(),
        "auc_std": cv_res_df['auc'].std(),
        "f1_mean": cv_res_df['f1'].mean(),
        "accuracy_mean": cv_res_df['accuracy'].mean(),
    })

l2_df = pd.DataFrame(l2_results)
print(l2_df)

   lambda  auc_mean   auc_std   f1_mean  accuracy_mean
0    0.00  0.726669  0.012469  0.538332       0.699721
1    0.01  0.726685  0.012477  0.538332       0.699721
2    0.10  0.726685  0.012462  0.539040       0.700610
3    1.00  0.726637  0.012441  0.539842       0.701495
4    5.00  0.726896  0.012612  0.536816       0.700610
5   10.00  0.727091  0.012511  0.533611       0.699725


In [56]:
for lam in [0, 0.1, 1]:
    m = LogisticRegressionScratch(learning_rate=0.1, n_iterations=10000, l2_lambda=lam)
    X2_train_s, X2_test_s, _, _ = standard_scale(X2_train, X2_test)
    m.fit(X2_train_s, y2_train)
    print(f"lambda={lam}: weights={np.round(m.w, 3)}")

lambda=0: weights=[0.086 0.155 0.075 0.18  0.774]
lambda=0.1: weights=[0.085 0.155 0.074 0.18  0.773]
lambda=1: weights=[0.085 0.155 0.073 0.179 0.767]


In [57]:
np.corrcoef(model_df_v2['conceded_avg_5'], model_df_v2['rank_diff'])

array([[ 1.        , -0.29309716],
       [-0.29309716,  1.        ]])

In [58]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

sklearn_model = LogisticRegression(penalty=None, max_iter=10000)
sklearn_model.fit(X2_train_s, y2_train)

print("Sklearn weights:", sklearn_model.coef_[0])
print("Sklearn bias:", sklearn_model.intercept_[0])
print("\nYour weights:  ", model_v2.w)
print("Your bias:     ", model_v2.b)

sklearn_proba = sklearn_model.predict_proba(X2_test_s)[:, 1]
print("\nSklearn test AUC:", roc_auc_score(y2_test, sklearn_proba))
print("Your test AUC:    ", roc_auc_score(y2_test, y2_proba))

Sklearn weights: [0.1049321  0.14321813 0.0945009  0.18397524 0.79437225]
Sklearn bias: -0.543392140460458

Your weights:   [0.08553233 0.15529436 0.07453158 0.17986969 0.77350062]
Your bias:      -0.5361789697119193

Sklearn test AUC: 0.7552890221555888
Your test AUC:     0.7559553556555056


C:\Users\bimar\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
